In [ ]:
# notebooks/eda.ipynb
# Run: jupyter notebook notebooks/eda.ipynb
# Or convert to script: jupyter nbconvert --to script notebooks/eda.ipynb

# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

PROCESSED_DIR = Path("../data/processed")
FIG_DIR = Path("../notebooks/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

ratings = pd.read_parquet(PROCESSED_DIR / "ratings.parquet")
items   = pd.read_parquet(PROCESSED_DIR / "items.parquet")

print(f"ratings shape : {ratings.shape}")
print(f"items shape   : {items.shape}")
ratings.head()

# ── Cell 2: Markdown summary — dataset overview ───────────────────────────────
"""
## Dataset Overview — MovieLens 100K

| Property | Value |
|----------|-------|
| Source | GroupLens Research, University of Minnesota |
| Users | 943 |
| Films | 1,682 |
| Ratings | 100,000 |
| Rating scale | 1–5 (integer) |
| Time span | Sep 1997 – Apr 1998 |

Each row in `ratings` is a single user–film interaction.
`items` contains film metadata: title, release year, and genre tags.

**Key questions this EDA answers:**
1. How active are users? (interaction distribution)
2. How popular are films? (long-tail effect)
3. How sparse is the interaction matrix? (impacts collaborative filtering)
4. Which genres dominate the catalogue?
5. How are ratings distributed? (bias toward positive ratings?)
6. Is there a temporal pattern in rating behaviour?
"""

In [ ]:
# ── Cell 3: Basic stats ───────────────────────────────────────────────────────
n_users   = ratings["user_id"].nunique()
n_items   = ratings["item_id"].nunique()
n_ratings = len(ratings)
sparsity  = 1 - n_ratings / (n_users * n_items)
density   = n_ratings / (n_users * n_items)

print("=" * 45)
print(f"  Users              : {n_users:>8,}")
print(f"  Films              : {n_items:>8,}")
print(f"  Ratings            : {n_ratings:>8,}")
print(f"  Possible pairs     : {n_users * n_items:>8,}")
print(f"  Sparsity           : {sparsity:>8.4f}  ({sparsity*100:.2f}%)")
print(f"  Density            : {density:>8.4f}  ({density*100:.2f}%)")
print(f"  Mean rating        : {ratings['rating'].mean():>8.3f}")
print(f"  Median rating      : {ratings['rating'].median():>8.1f}")
print(f"  Std rating         : {ratings['rating'].std():>8.3f}")
print("=" * 45)

# ── Cell 4: Markdown — sparsity interpretation ────────────────────────────────
"""
## Finding 1 — Matrix Sparsity

The interaction matrix is **93.70% sparse**: only ~6.3% of all possible
user–film pairs have a rating.

**Why this matters for modelling:**

- Pure collaborative filtering (user–user or item–item nearest neighbours)
  struggles with high sparsity because overlap between users is limited.
- Matrix factorisation (ALS) handles sparsity well — it learns dense latent
  representations that generalise across the sparse observed interactions.
- Content-based filtering is unaffected by sparsity since it operates on
  item metadata, not interaction data.
- This justifies the **hybrid approach**: content-based signals cover items
  and users with few interactions; collaborative signals dominate for
  well-represented users and items.
"""

In [ ]:
# ── Cell 5: User activity distribution ───────────────────────────────────────
user_activity = ratings.groupby("user_id")["item_id"].count().rename("n_ratings")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(user_activity, bins=50, color="#4C72B0", edgecolor="white", linewidth=0.5)
axes[0].set_title("User Activity Distribution", fontweight="bold")
axes[0].set_xlabel("Number of ratings per user")
axes[0].set_ylabel("Number of users")
axes[0].axvline(user_activity.median(), color="#DD8452", linestyle="--",
                label=f"Median = {user_activity.median():.0f}")
axes[0].axvline(user_activity.mean(), color="#55A868", linestyle="--",
                label=f"Mean = {user_activity.mean():.0f}")
axes[0].legend()

# Cumulative distribution
sorted_activity = user_activity.sort_values()
cumulative = np.arange(1, len(sorted_activity) + 1) / len(sorted_activity)
axes[1].plot(sorted_activity.values, cumulative * 100, color="#4C72B0", linewidth=2)
axes[1].set_title("Cumulative User Activity", fontweight="bold")
axes[1].set_xlabel("Number of ratings per user")
axes[1].set_ylabel("Cumulative % of users")
axes[1].axhline(80, color="#DD8452", linestyle="--", label="80th percentile")
axes[1].axvline(user_activity.quantile(0.8), color="#DD8452", linestyle="--")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "user_activity.png", dpi=150, bbox_inches="tight")
plt.show()

print(user_activity.describe().to_frame().T.to_markdown(index=False))

# ── Cell 6: Markdown — user activity findings ─────────────────────────────────
"""
## Finding 2 — User Activity (Long Tail)

| Stat | Value |
|------|-------|
| Min ratings per user | 20 |
| Median ratings per user | ~65 |
| Mean ratings per user | ~106 |
| Max ratings per user | 737 |

**Observations:**
- MovieLens 100K enforces a minimum of 20 ratings per user, which removes
  the extreme cold-start problem present in production systems.
- Despite the minimum, the distribution is **right-skewed**: a small number
  of power users account for a disproportionate share of ratings.
- The top 20% of users contribute roughly 50%+ of all ratings.
- In a real system, users below ~20 interactions are cold-start candidates
  where content-based or popularity fallback is necessary.
"""

In [ ]:
# ── Cell 7: Item popularity distribution ──────────────────────────────────────
item_popularity = ratings.groupby("item_id")["user_id"].count().rename("n_ratings")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-scale histogram to show the long tail
axes[0].hist(item_popularity, bins=60, color="#55A868", edgecolor="white", linewidth=0.5)
axes[0].set_title("Item Popularity Distribution (linear)", fontweight="bold")
axes[0].set_xlabel("Number of ratings per film")
axes[0].set_ylabel("Number of films")
axes[0].axvline(item_popularity.median(), color="#DD8452", linestyle="--",
                label=f"Median = {item_popularity.median():.0f}")
axes[0].legend()

axes[1].hist(item_popularity, bins=60, color="#55A868", edgecolor="white", linewidth=0.5,
             log=True)
axes[1].set_title("Item Popularity Distribution (log y-axis)", fontweight="bold")
axes[1].set_xlabel("Number of ratings per film")
axes[1].set_ylabel("Number of films (log scale)")

plt.tight_layout()
plt.savefig(FIG_DIR / "item_popularity.png", dpi=150, bbox_inches="tight")
plt.show()

# Top 10 most-rated films
top_items = (
    item_popularity
    .reset_index()
    .merge(items[["item_id", "title", "genres"]], on="item_id")
    .sort_values("n_ratings", ascending=False)
    .head(10)
)
print("\nTop 10 most-rated films:")
print(top_items[["title", "genres", "n_ratings"]].to_markdown(index=False))

# Items with fewer than 10 ratings
cold_items = (item_popularity < 10).sum()
print(f"\nFilms with fewer than 10 ratings: {cold_items} ({cold_items/len(item_popularity)*100:.1f}%)")

# ── Cell 8: Markdown — item popularity findings ───────────────────────────────
"""
## Finding 3 — Item Popularity (Severe Long Tail)

The item popularity distribution follows a classic **power law**:

- A small number of blockbusters (Star Wars, Fargo, Pulp Fiction) have
  hundreds of ratings each.
- The majority of films have fewer than 50 ratings.
- Roughly **30–40% of films have fewer than 10 ratings** — these are
  effectively invisible to collaborative filtering.

**Implications for our system:**
- The **popularity bias** problem: a naive recommender will over-recommend
  popular films because they dominate the interaction matrix signal.
- We address this with the **Bayesian score** in `PopularityRecommender`
  (penalises items with few ratings) and by including content-based signals
  in the hybrid model (which has no popularity bias).
- **Coverage metric** in evaluation will reveal whether the model only
  recommends the top-50 films or genuinely surfaces the long tail.
"""

In [ ]:
# ── Cell 9: Rating distribution ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw rating counts
rating_counts = ratings["rating"].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values,
            color=["#c44e52", "#dd8452", "#ccb974", "#55a868", "#4c72b0"],
            edgecolor="white", linewidth=0.8)
axes[0].set_title("Rating Distribution", fontweight="bold")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")
for i, (val, count) in enumerate(zip(rating_counts.index, rating_counts.values)):
    axes[0].text(val, count + 200, f"{count:,}", ha="center", va="bottom", fontsize=9)

# Rating distribution by genre (top 6 genres)
genre_cols = [
    "Action", "Comedy", "Drama", "Romance", "Thriller", "Sci-Fi"
]
# Rebuild genre flags from items
items_with_flags = pd.read_parquet(PROCESSED_DIR / "items.parquet")
genre_means = {}
for genre in genre_cols:
    genre_item_ids = []
    for _, row in items_with_flags.iterrows():
        if genre in row["genres"]:
            genre_item_ids.append(row["item_id"])
    if genre_item_ids:
        mean_r = ratings[ratings["item_id"].isin(genre_item_ids)]["rating"].mean()
        genre_means[genre] = mean_r

genre_df = pd.Series(genre_means).sort_values(ascending=True)
axes[1].barh(genre_df.index, genre_df.values, color="#4C72B0")
axes[1].set_title("Mean Rating by Genre (top 6)", fontweight="bold")
axes[1].set_xlabel("Mean Rating")
axes[1].axvline(ratings["rating"].mean(), color="#DD8452", linestyle="--",
                label=f"Overall mean = {ratings['rating'].mean():.2f}")
axes[1].legend()
axes[1].set_xlim(3.0, 4.5)

plt.tight_layout()
plt.savefig(FIG_DIR / "rating_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nRating value counts:")
print(rating_counts.to_frame().rename(columns={"rating": "count"}).to_markdown())